intphy

In [ ]:
# !wget https://storage.googleapis.com/representations4d/checkpoints/pretrain_rvm_large16_256_202497301.npz

In [ ]:
import glob
import math

import random
import os
from icecream import ic

import cv2
import einops
from PIL import Image
import scipy.io as sio
import mediapy as media
import numpy as np
import pandas as pd
import queue
import seaborn as sns
import tensorflow as tf
import tqdm
import jax
import sklearn
import matplotlib.pyplot as plt
import cv2

In [ ]:
from models.rvm_jax import build_model_L, build_model
masking_ratio = 0.95
# masking_ratio = 1
model = build_model_L(masking_ratio)
# model = build_model(masking_ratio)

In [ ]:
import flax

def merge(init_p, ckpt_p):
  """ckpt values win; init values fill the gaps. Recurses on nested dicts."""
  init_p = flax.core.unfreeze(init_p)
  out = {}
  for k, v in init_p.items():
    if k not in ckpt_p:
      out[k] = v                                  # new: decoder_proj
    elif isinstance(v, dict):
      out[k] = merge(v, ckpt_p[k])
    else:
      c = np.asarray(ckpt_p[k])
      assert c.shape == v.shape, f'{k}: ckpt {c.shape} vs model {v.shape}'
      out[k] = jnp.asarray(c)
  return out


def recover_tree(flat_dict):
  tree = {}
  for k, v in flat_dict.items():
    parts = k.split("/")
    node = tree
    for part in parts[:-1]:
      if part not in node:
        node[part] = {}
      node = node[part]
    node[parts[-1]] = v
  return tree

restored_params = recover_tree(np.load("/home/rvm/rvm_ckpts/pretrain_rvm_large16_256_202497301.npz", allow_pickle=False))
# restored_params = recover_tree(np.load("/home/rvm/rvm_ckpts/pretrain_rvm_small16_256_204031069.npz", allow_pickle = False))
count = sum([np.prod(v.shape) for v in jax.tree_util.tree_leaves(restored_params)])
print(f'number of params L = {count}')


dummy_src, dummy_tgt = np.random.randn(1, 4, 256, 256, 3), np.random.randn(1, 4, 256, 256, 3)
dummy_deltas = np.array([4, 5, 6, 7])

key = jax.random.PRNGKey(0)
rng, rng2 = jax.random.split(key)

init_params = model.init(
    {'params': rng, 'default': rng2},
    dummy_src, dummy_tgt, dummy_deltas,
    method=model.reconstruct,
)['params']
# params = merge(init_params, restored_params)



In [ ]:
rng_seed = 0
rng_key = {'default': jax.random.PRNGKey(rng_seed)}
@jax.jit
def forward(params, source, target, target_deltas):
    return model.apply(
        {'params': params},
        source, target, target_deltas,
        method=model.reconstruct,
        rngs=rng_key,
    )

In [ ]:
video_path = '/home/rvm/datasets/Main/Videos/c99d46743f5a5760705af190faa69e1bacae4b788c1e44c901c47b78de7ab7cc.mp4'
import cv2
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Cannot open video")

frames = []
while True:
    ret, frame = cap.read()
    if not ret: break
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (256, 256))
    frame = frame/255.0
    frames.append(frame)    

print(len(frames))
stride = 10
frames = frames[int(0.2 * len(frames)):]
frames = frames[::stride]
print(len(frames))


In [ ]:
context = 10
pred_future = 6

In [ ]:
source_frames = np.array(frames[:context])[None, ...]
target_frames = np.array(frames[context : context + pred_future])[None, ...]
target_deltas = np.array([11, 12, 13, 14, 15, 16])

output = forward(restored_params, source_frames, target_frames, target_deltas)
recon = output['reconstructed']

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

targets = target_frames[0]  # (6, 512, 512, 3)
preds = recon[0]            # (6, 512, 512, 3)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))

for i in range(6):
    target = np.asarray(targets[i])

    # If target is already black, this will just display it
    if target.max() > 1:
        target = target.astype(np.uint8)
    else:
        target = (np.clip(target, 0, 1) * 255).astype(np.uint8)

    axes[0, i].imshow(target)
    axes[0, i].set_title(f"Target {target_deltas[i]}")
    axes[0, i].axis("off")

    image = np.asarray(preds[i]).astype(np.float32)

    print(f"Pred {i}: {image.shape}, {image.min():.4f}, {image.max():.4f}")

    # [0, 1] -> [0, 255]
    image = np.clip(image, 0.0, 1.0)
    image = (image * 255.0).astype(np.uint8)

    # recon is RGB, matplotlib expects RGB
    axes[1, i].imshow(image)
    axes[1, i].set_title(f"Pred {target_deltas[i]}")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Target", fontsize=14)
axes[1, 0].set_ylabel("Pred", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
frame1, frame2 = source_frames[:, -2,...].squeeze(0), source_frames[:, -1,...].squeeze(0)
frame1 = (np.clip(frame1, 0, 1) * 255).astype(np.uint8)
frame2 = (np.clip(frame2, 0, 1) * 255).astype(np.uint8)

plt.subplot(1, 2, 1)
plt.imshow(frame1)
plt.subplot(1, 2, 2)
plt.imshow(frame2)

In [ ]:
output['decoded_representation'].shape

vjepa2.1

In [ ]:
import torch
import torch.nn.functional as F

from models.make_model_vj2_1 import vjepa2_1_vit_giant_384
from models.models_vj2.utils.multimask import MultiMaskWrapper
from models.models_vj2.utils.mask_utils import apply_masks

# vit_giant encoder alone is ~4GB fp32 -- doesn't fit this machine's 4GB VRAM, so run on CPU.
device = "cpu"
resolution = 384
frames_per_clip = 16    # total frames the encoder sees in one clip
nb_context_frames = 8   # first half is context; predictor guesses representations for the rest
checkpoint = "/home/mahesh/VJEPA2.1-Model/ckpts/2.1/vjepa2_1_vitg_384.pt"

In [ ]:
video_path = '/home/mahesh/VJEPA2.1-Model/datasets/IntPhys2/Main/Videos/c99d46743f5a5760705af190faa69e1bacae4b788c1e44c901c47b78de7ab7cc.mp4'
import cv2
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Cannot open video")

frames = []
while True:
    ret, frame = cap.read()
    if not ret: break
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (resolution, resolution))
    frame = frame/255.0
    frames.append(frame)    

print(len(frames))
stride = 10
frames = frames[int(0.2 * len(frames)):]
frames = frames[::stride]
print(len(frames))
assert len(frames) >= frames_per_clip, "not enough sampled frames for one clip"


In [ ]:
# the checkpoint dict also carries optimizer state (~8.5GB) alongside the encoder/predictor
# weights, so this load is slow and memory-heavy -- expected, not a bug.
encoder, predictor = vjepa2_1_vit_giant_384(checkpoint, num_frames=frames_per_clip)
encoder, predictor = encoder.to(device), predictor.to(device)
encoder.eval()
predictor.eval()

# predictor_embed/predictor_proj are sized for the 4-layer hierarchical concat, so both the
# context fed into the predictor and the targets compared against it must come from that
# hierarchical encoder output (same trick the IntPhys2 eval wrapper uses).
encoder.return_hierarchical = True
encoder = MultiMaskWrapper(encoder)
target_encoder = encoder  # single pretrained checkpoint, reused for context + target extraction

params

In [ ]:
e_count = sum(p.numel() for p in encoder.parameters())
p_count = sum(p.numel() for p in predictor.parameters())

print(f'encoder params {e_count}, predictor params {p_count}')

In [ ]:
patch_size = encoder.backbone.patch_size
tubelet_size = encoder.backbone.tubelet_size

def get_time_masks(n_timesteps, spatial_size, temporal_size, spatial_dim, temporal_dim):
    """Split a flattened (time-major) token sequence into a context prefix (first
    n_timesteps frames) and a prediction suffix (the remaining frames)."""
    x, y = spatial_dim
    t = temporal_dim
    num_patches_spatial = x / spatial_size[0] * y / spatial_size[1]
    num_patches_time = t / temporal_size
    patches_n_timesteps = int(num_patches_spatial * n_timesteps // temporal_size)
    patch_idcs = torch.arange(start=0, end=int(num_patches_spatial * num_patches_time), dtype=int)
    mask_enc = patch_idcs[:patches_n_timesteps]
    mask_pred = patch_idcs[patches_n_timesteps:]
    full_mask = patch_idcs
    return mask_enc, mask_pred, full_mask

m, m_, full_m = get_time_masks(
    nb_context_frames,
    spatial_size=(patch_size, patch_size),
    temporal_size=tubelet_size,
    spatial_dim=(resolution, resolution),
    temporal_dim=frames_per_clip,
)

masks_enc = [m.unsqueeze(0).to(device)]
masks_pred = [m_.unsqueeze(0).to(device)]
full_mask = [full_m.unsqueeze(0).to(device)]

grid_size = resolution // patch_size
grid_depth_future = (frames_per_clip - nb_context_frames) // tubelet_size
print(f"context tokens: {m.numel()}, prediction tokens: {m_.numel()}, "
      f"grid {grid_size}x{grid_size}, future temporal patches {grid_depth_future}")

In [ ]:
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1, 1)

clip_frames = np.array(frames[:frames_per_clip])                               # (T, H, W, 3) in [0, 1]
clip = torch.from_numpy(clip_frames).float().permute(3, 0, 1, 2).unsqueeze(0)  # (1, C, T, H, W)
clip = (clip - imagenet_mean) / imagenet_std
clip = clip.to(device)

with torch.no_grad():
    # target representations: encode the *whole* clip (context + future), then keep only the
    # future tokens. This is the JEPA target -- it never runs through the predictor.
    h_full = target_encoder(clip, full_mask)[0]
    targets = apply_masks(h_full, masks_pred, concat=False)[0]
    targets = F.layer_norm(targets, (targets.size(-1),))

    # predicted representations: encode only the context tokens, then let the predictor's
    # learnable mask tokens (+ positions) stand in for the future and predict their features.
    context = encoder(clip, masks_enc)[0]
    preds, _ = predictor(context, masks_enc[0], masks_pred[0], mod="video", mask_index=0)

print("targets", targets.shape, "preds", preds.shape)

In [ ]:
from pca_vis import pca_per_frame

n_per_frame = grid_size * grid_size
targets_feats = targets.reshape(1, grid_depth_future, n_per_frame, -1)
preds_feats = preds.reshape(1, grid_depth_future, n_per_frame, -1)

print("target PCA:")
target_imgs = pca_per_frame(targets_feats)   # (grid_depth_future, grid_size, grid_size, 3)
print("pred PCA:")
pred_imgs = pca_per_frame(preds_feats)

In [ ]:
upsample = 16
target_big = np.repeat(np.repeat(target_imgs, upsample, axis=1), upsample, axis=2)
pred_big = np.repeat(np.repeat(pred_imgs, upsample, axis=1), upsample, axis=2)

T = grid_depth_future
fig, axes = plt.subplots(2, T, figsize=(3 * T, 6))
axes = np.atleast_2d(axes)
for t in range(T):
    start = nb_context_frames + t * tubelet_size

    axes[0, t].imshow(target_big[t])
    axes[0, t].set_title(f"Target frames {start}-{start + tubelet_size - 1}")
    axes[0, t].axis("off")

    axes[1, t].imshow(pred_big[t])
    axes[1, t].set_title(f"Pred frames {start}-{start + tubelet_size - 1}")
    axes[1, t].axis("off")

axes[0, 0].set_ylabel("Target", fontsize=14)
axes[1, 0].set_ylabel("Pred", fontsize=14)

plt.tight_layout()
plt.show()